# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ABK998/flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os, sys, subprocess
from google.colab import userdata
import duckdb

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ABK998/flyrank-ML"
REPO_DIR = "flyrank-ML"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

# Set up DuckDB with Hugging Face HTTPFS access
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
);
""")
print("DuckDB initialized with HF token.")

DuckDB initialized with HF token.


## 1. Unit of analysis + time window

1. Unit of Analysis (Grain): One row represents one pseudonymized content item (content_hash_id) for a specific client (client_hash_id) evaluated over a historical time window.

2. Tables Used: dim_content, dim_clients, and the mid-panel monthly partition of fact_content_daily_performance (month=2026-03).

3. Time Window: Historical feature window of March 2026 (2026-03-01 to 2026-03-31) to iterate safely without peeking into the final sealed test month (June 2026).

4. Prediction Target / Proxy: Supervised priority ranking score predicting traffic decline (trend_direction == 'down' or forward-window click loss).

5. Deliberately Excluded: trend_pct, trend_direction, raw identifiers, and product-level decision flags (health_score, priority_score, action_type).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 3. Verify it with queries (grain, counts, missing values, windows)



In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1: Grain verification (check for uniqueness on grain columns)
q1 = """
SELECT content_hash_id, COUNT(*) as cnt
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
GROUP BY content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
"""
print("Query 1 — Uniqueness Check (Should be empty):")
print(con.execute(q1).df())

# Query 2: Row count and date span on mid-panel month (month=2026-03)
q2 = """
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS total_daily_rows,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet';
"""
print("\nQuery 2 — Date Span and Volume for 2026-03:")
print(con.execute(q2).df())

# Query 3: Availability check using IS TRUE
q3 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    ROUND(COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS pct_ga4_available
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet';
"""
print("\nQuery 3 — GA4 Availability Check:")
print(con.execute(q3).df())


Query 1 — Uniqueness Check (Should be empty):
Empty DataFrame
Columns: [content_hash_id, cnt]
Index: []

Query 2 — Date Span and Volume for 2026-03:
    min_date   max_date  total_daily_rows  distinct_content_items
0 2026-03-01 2026-03-31           9841378                  331437

Query 3 — GA4 Availability Check:
   total_rows  ga4_available_rows  pct_ga4_available
0     9841378              413966               4.21


## 4. Data limits

total_impressions_30d: Aggregated sum of impressions during the 30-day feature window; knowable at decision time.

total_clicks_30d: Aggregated sum of clicks during the 30-day feature window; knowable at decision time.

avg_gsc_position: Mean search position during the feature window; knowable at decision time.

active_days_count: Count of days with non-zero impressions; reflects consistency prior to decision time.

content_age_days: Difference between feature cutoff date and content_created_at; fixed metadata knowable at decision time.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.